In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import log_loss
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC 
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
import os

lda = LinearDiscriminantAnalysis()
lr = LogisticRegression(penalty=None)
scaler = StandardScaler().set_output(transform='pandas')
svm = SVC(kernel="linear", probability=True, random_state=25)
pipe_svm = Pipeline([('SCL',scaler),('SVM',svm)])
dtc = DecisionTreeClassifier(random_state=25)
gbm = LGBMClassifier(random_state=25, verbosity=-1)
rf = RandomForestClassifier(random_state=25)

os.chdir("C:/Python/Cases/Glass Identification")
glass = pd.read_csv("Glass.csv")
le = LabelEncoder()
glass['Type'] = le.fit_transform( glass['Type'] )
X, y = glass.drop('Type', axis=1), glass['Type']

X_train,X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=25, stratify=y)

import warnings
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings("ignore", category=ConvergenceWarning, module="sklearn")

stack = StackingClassifier(estimators=[('LDA',lda),('LR',lr),('SVM',pipe_svm),('TREE',dtc)],
                           final_estimator=rf, passthrough=True)
stack.fit(X_train, y_train)
y_pred_prob = stack.predict_proba(X_test)
print(log_loss(y_test, y_pred_prob))

stack = StackingClassifier(estimators=[('LDA',lda),('LR',lr),('SVM',pipe_svm),('TREE',dtc)],
                           final_estimator=gbm, passthrough=True)
stack.fit(X_train, y_train)
y_pred_prob = stack.predict_proba(X_test)
print(log_loss(y_test, y_pred_prob))

